In [ ]:
import sys, os
# Adjust depth based on notebook location relative to project root
_src_path = os.path.normpath(os.path.join(os.getcwd(), '..', '..', 'src'))
if _src_path not in sys.path:
    sys.path.insert(0, _src_path)
from project_config import load_config

config    = load_config()
base_dir  = config['base_dir']
well_list = config['well_list']
well_meta = config.get('well_metadata', {})

print(f'Active experiment: {config.get("experiment_name", config.get("experiment_key", "?"))}')
print(f'Base dir: {base_dir}')

In [3]:
import os
import json
import numpy as np
import pandas as pd
import anndata as ad
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display

pio.renderers.default = "notebook_connected"

# Load the adata file that already has the ORIGINAL saved PHATE coordinates
adata = ad.read_h5ad(
    r'D:\Lauryn\Liposarcoma_2025\LPS_246_Abema_July_2025\standard_adata_sub__control_abema_PHATE.h5ad'
)

# Confirm PHATE coordinates exist
if "X_phate" not in adata.obsm:
    raise ValueError("No saved PHATE coordinates found in adata.obsm['X_phate']")

print("Loaded adata:", adata.shape)
print("Saved PHATE shape:", adata.obsm["X_phate"].shape)
print("Example features:", list(adata.var_names[:10]))

Loaded adata: (10500, 38)
Saved PHATE shape: (10500, 3)
Example features: ['nuc_area', 'orientation', 'major_axis_length', 'minor_axis_length', 'R0_DNA_nuc_mean', 'R0_pRb_nuc_mean', 'R0_Rb_nuc_mean', 'R0_CDK2_nuc_mean', 'R1_CDK4_nuc_mean', 'R1_p53_nuc_mean']


In [4]:
# Choose any feature for preview while picking the orientation
feature_to_preview = "R0_pRb_nuc_mean"

if feature_to_preview not in adata.var_names:
    raise ValueError(f"{feature_to_preview} not found in adata.var_names")

# Build dataframe from saved PHATE coordinates
phate_df = pd.DataFrame(
    adata.obsm["X_phate"],
    columns=["PHATE1", "PHATE2", "PHATE3"],
    index=adata.obs_names
)

# Add preview feature values
feat_idx = adata.var_names.get_loc(feature_to_preview)
vals = adata.X[:, feat_idx]
if hasattr(vals, "toarray"):
    vals = vals.toarray().ravel()
else:
    vals = np.asarray(vals).ravel()

phate_df["feature_value"] = vals

vmin = np.nanpercentile(vals, 1)
vmax = np.nanpercentile(vals, 99)

base_fig = px.scatter_3d(
    phate_df,
    x="PHATE1",
    y="PHATE2",
    z="PHATE3",
    color="feature_value",
    color_continuous_scale="viridis",
    range_color=[vmin, vmax],
    opacity=0.8,
    title=f"Rotate this to choose final manuscript orientation: {feature_to_preview}"
)

base_fig.update_traces(marker=dict(size=2))

base_fig.update_layout(
    width=1400,   # increase this (try 900–1400)
    height=800,
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="white",
        uirevision="keep_camera"
    ),
    paper_bgcolor="white",
    margin=dict(l=0, r=0, b=0, t=40),
    coloraxis_colorbar=dict(title=feature_to_preview)
)

fig = go.FigureWidget(base_fig)

button = widgets.Button(description="Print current camera", button_style="info")
output = widgets.Output()

def print_camera(_):
    with output:
        output.clear_output()
        cam = fig.layout.scene.camera.to_plotly_json()
        print("Copy this entire block and paste it into saved_camera in Block 3:\n")
        print("saved_camera = dict(")
        print(f"    center=dict(x={cam['center']['x']}, y={cam['center']['y']}, z={cam['center']['z']}),")
        print(f"    eye=dict(x={cam['eye']['x']}, y={cam['eye']['y']}, z={cam['eye']['z']}),")
        print(f"    up=dict(x={cam['up']['x']}, y={cam['up']['y']}, z={cam['up']['z']})")
        print(")")

button.on_click(print_camera)

display(button)
display(output)
display(fig)

Button(button_style='info', description='Print current camera', style=ButtonStyle())

Output()

FigureWidget({
    'data': [{'hovertemplate': ('PHATE1=%{x}<br>PHATE2=%{y}<br>' ... '%{marker.color}<extra></extra>'),
              'legendgroup': '',
              'marker': {'color': array([ 0.7061219 ,  0.619152  ,  2.148261  , ..., -0.80049586, -0.7637346 ,
                                         -0.6908807 ], dtype=float32),
                         'coloraxis': 'coloraxis',
                         'opacity': 0.8,
                         'size': 2,
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': '',
              'scene': 'scene',
              'showlegend': False,
              'type': 'scatter3d',
              'uid': 'b1e3b803-8b65-4266-a02a-cf3ad7dd283b',
              'x': array([-0.00745088, -0.00069496,  0.04549449, ..., -0.01701893, -0.00403978,
                          -0.0077623 ]),
              'y': array([ 0.00346102,  0.00807816, -0.00273382, ..., -0.00249552,  0.00395266,
                           0.00144987

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Output folder
output_dir = os.path.join(base_dir, 'PHATE_feature_PDFs_matplotlib')
os.makedirs(output_dir, exist_ok=True)

# Features to plot
features_to_plot = [
    "R2_cycB1_nuc_mean",
    "nuc_area",
    "R0_pRb_nuc_mean",
    "R0_CDK2_nuc_mean",
    "R3_Ki67_nuc_mean"
]

missing_features = [f for f in features_to_plot if f not in adata.var_names]
if missing_features:
    print("These features were not found and will be skipped:")
    for f in missing_features:
        print(" -", f)

features_to_plot = [f for f in features_to_plot if f in adata.var_names]

if len(features_to_plot) == 0:
    raise ValueError("None of the requested features were found in adata.var_names")

# PHATE coordinates
X_phate = adata.obsm["X_phate"]
x = X_phate[:, 0]
y = X_phate[:, 1]
z = X_phate[:, 2]

# Paste your Plotly camera coordinates here
saved_camera = dict(
    center=dict(x=0, y=0, z=0),
    eye=dict(x=-0.9983088867760009, y=-1.4956017958896348, z=-1.2058418779905435),
    up=dict(x=0, y=0, z=1)
)

# Convert Plotly camera eye position to approximate matplotlib elev/azim
eye_x = saved_camera["eye"]["x"]
eye_y = saved_camera["eye"]["y"]
eye_z = saved_camera["eye"]["z"]

r_xy = np.sqrt(eye_x**2 + eye_y**2)
azim = np.degrees(np.arctan2(eye_y, eye_x))
elev = np.degrees(np.arctan2(eye_z, r_xy))

print(f"Using approximate matplotlib view: elev={elev:.2f}, azim={azim:.2f}")

for feature_to_color in features_to_plot:
    print(f"Saving {feature_to_color}...")

    feat_idx = adata.var_names.get_loc(feature_to_color)
    vals = adata.X[:, feat_idx]
    if hasattr(vals, "toarray"):
        vals = vals.toarray().ravel()
    else:
        vals = np.asarray(vals).ravel()

    vmin = np.nanpercentile(vals, 1)
    vmax = np.nanpercentile(vals, 99)

    fig = plt.figure(figsize=(8, 8), dpi=300)
    ax = fig.add_subplot(111, projection='3d')

    sc = ax.scatter(
        x, y, z,
        c=vals,
        cmap='viridis',
        s=4,
        alpha=0.8,
        vmin=vmin,
        vmax=vmax,
        linewidths=0
    )

    ax.view_init(elev=elev, azim=azim)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_title(feature_to_color, fontsize=14)

    # Remove panes / grid for cleaner manuscript look
    ax.grid(False)
    try:
        ax.xaxis.pane.fill = False
        ax.yaxis.pane.fill = False
        ax.zaxis.pane.fill = False
        ax.xaxis.pane.set_edgecolor('white')
        ax.yaxis.pane.set_edgecolor('white')
        ax.zaxis.pane.set_edgecolor('white')
    except:
        pass

    # Hide axis lines if possible
    try:
        ax.w_xaxis.line.set_color((1, 1, 1, 0))
        ax.w_yaxis.line.set_color((1, 1, 1, 0))
        ax.w_zaxis.line.set_color((1, 1, 1, 0))
    except:
        pass

    cbar = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.02)
    cbar.set_label(feature_to_color, fontsize=12)

    safe_name = "".join(c if c.isalnum() or c in ("_", "-", ".") else "_" for c in feature_to_color)
    save_path = os.path.join(output_dir, f"{safe_name}.pdf")
    plt.savefig(save_path, bbox_inches='tight')
    plt.close(fig)

print(f"Done. Saved PDFs to: {output_dir}")

In [6]:
#run this block if you want to test different dot sizes:
for s in [2, 3, 4, 5]:
    fig = px.scatter_3d(
        phate_df,
        x="PHATE1",
        y="PHATE2",
        z="PHATE3",
        color="feature_value",
        color_continuous_scale="viridis",
        opacity=0.8,
        title=f"Dot size = {s}"
    )

    fig.update_traces(marker=dict(size=s))
    fig.show()